In [1]:
!pip install kaggle --upgrade -q
print("Kaggle upgraded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 2.0 MB/s eta 0:00:00
Kaggle upgraded


In [2]:
!pip install kagglehub -q

In [3]:
import pandas as pd
import requests
import os
import kagglehub
import shutil
import gzip

### **Kaggle dataset - USA Real Estate Dataset**

In [4]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
from google.colab import files
files.upload()

### How to correctly upload `kaggle.json`

1.  Run the cell below that contains `files.upload()`.
2.  A button named `Choose Files` will appear. Click on it.
3.  Select the `kaggle.json` file from your local machine.
4.  Once the upload is complete, you should see output similar to `Uploaded file: kaggle.json`. You can then proceed to run the next cells for configuring the Kaggle CLI.

In [ ]:
from google.colab import files
files.upload()


*   **Configuring CLI**







In [ ]:
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle CLI ready")



* **Download Dataset**


In [ ]:
!pip install kagglehub -q

import kagglehub
import shutil
import os

# Download using kagglehub
path = kagglehub.dataset_download(
    "ahmedshahriarsakib/usa-real-estate-dataset"
)

print(f"Downloaded to: {path}")
print("Files:", os.listdir(path))

In [ ]:
file_path = '/root/.cache/kagglehub/datasets/ahmedshahriarsakib/usa-real-estate-dataset/versions/25/realtor-data.zip.csv'

# Check file size
size = os.path.getsize(file_path)
print(f"File size: {size / (1024*1024):.1f} MB")

# Try to read first few lines raw
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 3:
            break

In [ ]:
file_path = '/root/.cache/kagglehub/datasets/ahmedshahriarsakib/usa-real-estate-dataset/versions/25/realtor-data.zip.csv'

df = pd.read_csv(file_path)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumns:\n{df.columns.tolist()}")

In [ ]:
dest = '/content/drive/MyDrive/Project1_RE_Price_Predictor/data/raw/realtor-data.csv'
shutil.copy(file_path, dest)
print(f"Saved to Drive: realtor-data.csv")
print(f"File size: {os.path.getsize(dest) / (1024*1024):.1f} MB")

### **Zillow — 2 Files**




* **Download ZHVI**



In [ ]:
dest = '/content/drive/MyDrive/Project1_RE_Price_Predictor/data/raw/'

# Zillow ZHVI All Homes Time Series
url = 'https://files.zillowstatic.com/research/public_csvs/zhvi/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv'

response = requests.get(url)
file_path = dest + 'zillow_zhvi.csv'

with open(file_path, 'wb') as f:
    f.write(response.content)

print(f"Downloaded: zillow_zhvi.csv")
print(f"Size: {os.path.getsize(file_path) / (1024*1024):.1f} MB")

# Quick look
df_zhvi = pd.read_csv(file_path)
print(f"Rows: {df_zhvi.shape[0]:,}")
print(f"Columns: {df_zhvi.shape[1]}")
df_zhvi.head(3)



*  **Download Zillow Inventory**



In [ ]:
# Zillow For-Sale Inventory
url = 'https://files.zillowstatic.com/research/public_csvs/invt_fs/Zip_invt_fs_uc_sfrcondo_sm_month.csv'

file_path = dest + 'zillow_inventory.csv'

response = requests.get(url)

with open(file_path, 'wb') as f:
    f.write(response.content)

print(f"Downloaded: zillow_inventory.csv")
print(f"Size: {os.path.getsize(file_path) / (1024*1024):.1f} MB")

df_inv = pd.read_csv(file_path)
print(f"Rows: {df_inv.shape[0]:,}")
print(f"Columns: {df_inv.shape[1]}")
df_inv.head(3)

In [ ]:
print("Files in raw folder:")
for f in sorted(os.listdir(dest)):
    size = os.path.getsize(dest + f) / (1024*1024)
    print(f"  {f}  ({size:.1f} MB)")

### **Download Redfin Market Tracker**

In [ ]:
dest = '/content/drive/MyDrive/Project1_RE_Price_Predictor/data/raw/'

url = 'https://redfin-public-data.s3.us-west-2.amazonaws.com/redfin_market_tracker/zip_code_market_tracker.tsv000.gz'

response = requests.get(url)
file_path = dest + 'redfin_market_tracker.gz'

with open(file_path, 'wb') as f:
    f.write(response.content)

print(f"Downloaded: redfin_market_tracker.gz")
print(f"Size: {os.path.getsize(file_path) / (1024*1024):.1f} MB")

In [ ]:
dest = '/content/drive/MyDrive/Project1_RE_Price_Predictor/data/raw/'
extracted_path = dest + 'redfin_zip_market_tracker.tsv'

# File already extracted in a previous run and still present on Drive,
# so we read it directly instead of re-extracting the (now-deleted) .gz source.

# Get columns without loading the full 4.48GB file into memory
cols = pd.read_csv(extracted_path, sep='\t', nrows=0).columns.tolist()
print(f"Columns: {len(cols)}")
print(f"\nColumns: {cols}")

# Efficient row count as a single expression - streams the file, low memory
row_count = sum(1 for _ in open(extracted_path)) - 1
print(f"Rows: {row_count:,}")

# Lightweight preview - only reads first 3 rows
df_redfin_preview = pd.read_csv(extracted_path, sep='\t', nrows=3)
df_redfin_preview

In [ ]:
df_redfin_preview.columns

In [ ]:
os.remove(file_path)
print("gz file deleted — keeping clean TSV only")

print("\nFiles in raw folder:")
for f in sorted(os.listdir(dest)):
  size = os.path.getsize(dest + f) / (1024*1024)
  print(f"  {f} ({size:.1f} MB)")

In [ ]:
import os
import pandas as pd

dest = '/content/drive/MyDrive/Project1_RE_Price_Predictor/data/raw/'
tsv_path = dest + 'redfin_zip_market_tracker.tsv'
csv_path = dest + 'redfin_market_tracker_trimmed.csv'
chunk_size = 200000
total_rows = 0
n_cols = None

cols_needed = [
    'PERIOD_BEGIN', 'PERIOD_END', 'REGION_TYPE', 'REGION',
    'CITY', 'STATE', 'STATE_CODE', 'PROPERTY_TYPE',
    'MEDIAN_SALE_PRICE', 'MEDIAN_LIST_PRICE', 'MEDIAN_PPSF',
    'MEDIAN_DOM', 'HOMES_SOLD', 'INVENTORY', 'MONTHS_OF_SUPPLY',
    'AVG_SALE_TO_LIST', 'SOLD_ABOVE_LIST', 'PARENT_METRO_REGION',
]

for i, chunk in enumerate(pd.read_csv(tsv_path, sep='\t', usecols=cols_needed, chunksize=chunk_size)):
    chunk.to_csv(csv_path, mode='w' if i == 0 else 'a', header=(i == 0), index=False)
    total_rows += len(chunk)
    n_cols = chunk.shape[1]

print(f"Converted to CSV (trimmed to {n_cols} columns)")
print(f"Rows:    {total_rows:,}")
print(f"Columns: {n_cols}")
print(f"New file size: {os.path.getsize(csv_path) / (1024*1024):.1f} MB")

In [ ]:
os.remove(dest + 'redfin_zip_market_tracker.tsv')
os.remove(dest + 'redfin_market_tracker.gz')

### **Download Realtor.com Zip Level Inventory**

In [ ]:
import requests
import os

dest = '/content/drive/MyDrive/Project1_RE_Price_Predictor/data/raw/'

# Alternative Realtor.com URL
url = 'https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_Zip_History.csv'

response = requests.get(url)
print(f"Status code: {response.status_code}")
print(f"Content size: {len(response.content) / (1024*1024):.1f} MB")

if response.status_code == 200:
    file_path = dest + 'realtor_zip_inventory.csv'
    with open(file_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded successfully")
else:
    print("Failed — trying next URL")

In [ ]:
df_realtor = pd.read_csv(file_path)

print(f"Rows:    {df_realtor.shape[0]:,}")
print(f"Columns: {df_realtor.shape[1]}")
print(f"\nColumns: {df_realtor.columns.tolist()}")
df_realtor.head(3)

In [ ]:
print("Files in raw folder:")
for f in sorted(os.listdir(dest)):
    if f != 'processed':
        size = os.path.getsize(dest + f) / (1024*1024)
        print(f"  {f}  ({size:.1f} MB)")

In [ ]:
print(df_realtor['month_date_yyyymm'].min(), df_realtor['month_date_yyyymm'].max())
print(df_realtor['month_date_yyyymm'].nunique())
print(df_realtor['month_date_yyyymm'].value_counts())
